# 15 · Synthesize All Neighbourhoods

Generate synthetic populations for **every** neighbourhood in Gothenburg, produce per-area error reports, and a consolidated summary.

⏱️ This notebook processes all 96 areas and takes several minutes to run.

In [6]:
import logging
import os
import json
import traceback
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

from gbgsynth import GbgSynth

logging.basicConfig(level=logging.WARNING)

### Configuration

In [7]:
YEAR = 2023
OUTPUT_DIR = Path.cwd().parent / "output"
POP_DIR = OUTPUT_DIR / "populations"
REPORT_DIR = OUTPUT_DIR / "reports"

POP_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# Areas to skip (zero population or no geographic area)
SKIP_AREAS = {"199"}  # Ospecificerat

print(f"Year:   {YEAR}")
print(f"Output: {OUTPUT_DIR}")

Year:   2023
Output: /Users/ssanjay/GitHub/GbgSynth/output


### Discover all areas

In [8]:
city = GbgSynth(year=YEAR)
all_areas = city.get_all_areas()
print(f"Found {len(all_areas)} neighbourhoods")

Found 96 neighbourhoods


### Helper — generate a per-area error report

In [9]:
def generate_error_report(area, comparisons, execution_time, error=None):
    """Build a text error report for one neighbourhood."""
    lines = []
    ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    lines.append("=" * 80)
    lines.append("SYNTHETIC POPULATION ERROR REPORT")
    lines.append("=" * 80)

    if area:
        lines.append(f"Neighbourhood: {area.area_name}")
        lines.append(f"Area Code:     {area.area_code}")
        lines.append(f"Year:          {area.year}")
    else:
        lines.append("Neighbourhood: GENERATION FAILED")

    lines.append(f"Generated:     {ts}")
    lines.append(f"Execution Time: {execution_time:.2f}s")
    lines.append("")

    if error:
        lines += ["!" * 80, "GENERATION ERROR", "!" * 80, error, "!" * 80, ""]
        return "\n".join(lines)

    # Summary statistics
    stats = area.get_summary_statistics()
    lines.append("-" * 80)
    lines.append("SUMMARY STATISTICS")
    lines.append("-" * 80)
    for label, key in [
        ("Total Population", "total_population"),
        ("Total Households", "total_households"),
        ("Average HH Size", "avg_household_size"),
        ("Total Cars", "total_cars"),
    ]:
        v = stats.get(key, "N/A")
        lines.append(f"  {label + ':':30s} {v:>10,}" if isinstance(v, int) else f"  {label + ':':30s} {v:>10.2f}")
    lines.append("")

    # Per-dimension tables
    for key, data in comparisons.items():
        if key == "overall" or not data or "comparison" not in data:
            continue
        lines.append("-" * 80)
        lines.append(f"MARGINAL COMPARISON: {data['name'].upper()}")
        lines.append("-" * 80)
        lines.append(f"{'Category':<35} {'Census':>10} {'Synth':>10} {'Diff':>10} {'Error%':>10}")
        lines.append("-" * 75)
        for row in sorted(data["comparison"], key=lambda r: abs(r["error_pct"]), reverse=True):
            marker = " ⚠️" if abs(row["error_pct"]) > 20 else ""
            lines.append(
                f"{str(row['category'])[:35]:<35} {row['actual']:>10,} {row['synth']:>10,} "
                f"{row['diff']:>+10,} {row['error_pct']:>9.1f}%{marker}"
            )
        lines.append("")

    # Overall fit
    if "overall" in comparisons:
        ov = comparisons["overall"]
        lines.append("=" * 80)
        lines.append("OVERALL FIT")
        lines.append("=" * 80)
        lines.append(f"  RMSE:        {ov['rmse']:.2f}")
        lines.append(f"  MAE:         {ov['mae']:.2f}")
        lines.append(f"  Correlation: {ov['correlation']:.4f}")
        lines.append("")

    return "\n".join(lines)

### Synthesize every neighbourhood

Each area is synthesized, saved to CSV, and an error report is written.

In [10]:
all_results = []
successful, failed, skipped = 0, 0, 0
total = len(all_areas)

for idx, (code, name) in enumerate(all_areas.items(), 1):
    safe_name = name.replace(" ", "_").replace("/", "-")

    if code in SKIP_AREAS:
        print(f"[{idx:>3}/{total}] ⏭️  {name} — skipped")
        skipped += 1
        continue

    result = dict(area_code=code, area_name=name, status="pending",
                  execution_time=0, error=None, stats=None, comparisons=None)
    t0 = datetime.now()

    try:
        area = city.get_area(code)
        area.generate()
        dt = (datetime.now() - t0).total_seconds()
        result["execution_time"] = dt

        stats = area.get_summary_statistics()
        comparisons = area.compare_to_marginals(print_report=False)
        result.update(stats=stats, comparisons=comparisons, status="success")

        # Save CSVs
        area.save_to_csv(str(POP_DIR / f"{code}_{safe_name}_individuals.csv"))
        area.save_households_to_csv(str(POP_DIR / f"{code}_{safe_name}_households.csv"))

        # Save error report
        report = generate_error_report(area, comparisons, dt)
        (REPORT_DIR / f"{code}_{safe_name}_error_report.txt").write_text(report, encoding="utf-8")

        successful += 1
        ov = comparisons.get("overall", {})
        corr = ov.get("correlation", 0)
        mape = ov.get("mape", 0)
        print(f"[{idx:>3}/{total}] ✅ {name:<25s}  pop={stats['total_population']:>5,}  "
              f"MAPE={mape:>5.1f}%  r={corr:.4f}  ({dt:.1f}s)")

    except Exception as e:
        dt = (datetime.now() - t0).total_seconds()
        result.update(execution_time=dt, status="failed",
                      error=f"{type(e).__name__}: {e}\n{traceback.format_exc()}")
        report = generate_error_report(None, {}, dt, result["error"])
        (REPORT_DIR / f"{code}_{safe_name}_error_report.txt").write_text(report, encoding="utf-8")
        failed += 1
        print(f"[{idx:>3}/{total}] ❌ {name:<25s}  {e}")

    all_results.append(result)

print(f"\nDone: {successful} succeeded, {failed} failed, {skipped} skipped")

[  1/96] ✅ 101 Kungsladugård          pop=10,815  MAPE=  8.1%  r=0.9961  (1.4s)
[  2/96] ✅ 102 Sanna                  pop=3,346  MAPE=  7.2%  r=0.9991  (0.3s)


[  3/96] ✅ 103 Majorna                pop=10,955  MAPE=  7.5%  r=0.9986  (0.9s)


[  4/96] ✅ 104 Stigberget             pop=7,781  MAPE=  9.8%  r=0.9980  (1.4s)


[  5/96] ✅ 105 Masthugget             pop=11,431  MAPE=  8.4%  r=0.9983  (5.9s)


[  6/96] ✅ 106 Änggården              pop=1,473  MAPE= 11.3%  r=0.9706  (1.2s)


...

[ 89/96] ✅ 702 Länsmansgården         pop=5,891  MAPE= 11.1%  r=0.9770  (5.4s)


[ 90/96] ✅ 703 Svartedalen            pop=4,591  MAPE= 10.5%  r=0.9808  (9.4s)
[ 91/96] ✅ 704 Hjuvik                 pop=7,426  MAPE=  9.3%  r=0.9972  (2.4s)


[ 92/96] ✅ 705 Nolered                pop=10,999  MAPE=  8.3%  r=0.9991  (7.8s)
[ 93/96] ✅ 706 Björlanda              pop=8,956  MAPE=  9.9%  r=0.9973  (2.7s)
[ 94/96] ❌ 707 Arendal                Total of weights must be greater than zero


[ 95/96] ✅ 708 Södra Biskopsgården    pop=8,373  MAPE= 10.8%  r=0.9709  (13.0s)


[ 96/96] ✅ 709 Jättesten              pop=7,746  MAPE=  9.2%  r=0.9896  (1.7s)

Done: 95 succeeded, 1 failed, 0 skipped


### Consolidated summary report

In [11]:
rows = []
for r in all_results:
    row = dict(area_code=r["area_code"], area_name=r["area_name"],
               status=r["status"], time_sec=r["execution_time"])
    if r["stats"]:
        row.update(population=r["stats"]["total_population"],
                   households=r["stats"]["total_households"],
                   avg_hh_size=r["stats"]["avg_household_size"],
                   cars=r["stats"]["total_cars"])
    if r["comparisons"] and "overall" in r["comparisons"]:
        ov = r["comparisons"]["overall"]
        row.update(rmse=ov.get("rmse"), mae=ov.get("mae"),
                   correlation=ov.get("correlation"), mape=ov.get("mape"))
        corr = ov.get("correlation", 0)
        row["grade"] = ("A" if corr >= 0.99 else "B" if corr >= 0.98
                        else "C" if corr >= 0.97 else "D" if corr >= 0.95 else "F")
    rows.append(row)

summary_df = pd.DataFrame(rows)
summary_df.to_csv(OUTPUT_DIR / "summary_report.csv", index=False)
print(f"Saved summary_report.csv ({len(summary_df)} rows)")
summary_df.head(10)

Saved summary_report.csv (96 rows)


,area_code,area_name,status,time_sec,population,households,avg_hh_size,cars,rmse,mae,correlation,mape,grade
0,101,101 Kungsladugård,success,1.423866,10815.0,6064.0,1.783476,2610.0,218.999558,82.741935,0.996134,8.106963,A
1,102,102 Sanna,success,0.307073,3346.0,2212.0,1.512658,603.0,32.639774,15.354839,0.999112,7.211329,A
2,103,103 Majorna,success,0.938194,10955.0,6264.0,1.748883,2689.0,138.477784,57.387097,0.998631,7.491437,A
3,104,104 Stigberget,success,1.407497,7781.0,4183.0,1.860148,1589.0,115.567464,47.322581,0.998048,9.760030,A
4,105,105 Masthugget,success,5.918959,11431.0,6513.0,1.755105,2529.0,165.336997,72.193548,0.998320,8.404985,A
5,106,106 Änggården,success,1.192807,1473.0,665.0,2.215038,383.0,78.602922,27.774194,0.970592,11.323781,C
6,107,107 Haga,success,0.194401,3808.0,2044.0,1.863014,740.0,44.003299,17.838710,0.998909,8.693281,A
7,108,108 Annedal,success,0.248512,4266.0,2590.0,1.647104,928.0,65.125685,27.032258,0.998183,8.789592,A
8,109,109 Olivedal,success,1.703088,11075.0,6190.0,1.789176,2605.0,151.213052,64.096774,0.998473,8.423261,A
9,110,110 Krokslätt,success,4.430360,16399.0,10058.0,1.630443,3128.0,321.448636,139.290323,0.996684,7.816772,A


### Detailed per-category comparison CSV

In [12]:
detail_rows = []
for r in all_results:
    if not r["comparisons"]:
        continue
    for dim_key, data in r["comparisons"].items():
        if dim_key == "overall" or not data or "comparison" not in data:
            continue
        for comp in data["comparison"]:
            detail_rows.append(dict(
                area_code=r["area_code"], area_name=r["area_name"],
                dimension=data["name"], category=comp["category"],
                census=comp["actual"], synth=comp["synth"],
                diff=comp["diff"], error_pct=comp["error_pct"]
            ))

detail_df = pd.DataFrame(detail_rows)
detail_df.to_csv(OUTPUT_DIR / "detailed_comparisons.csv", index=False)
print(f"Saved detailed_comparisons.csv ({len(detail_df)} rows)")

Saved detailed_comparisons.csv (2945 rows)


### JSON results

In [13]:
json_out = []
for r in all_results:
    entry = dict(area_code=r["area_code"], area_name=r["area_name"],
                 status=r["status"], execution_time=r["execution_time"],
                 error=r["error"])
    if r["stats"]:
        entry["stats"] = r["stats"]
    if r["comparisons"] and "overall" in r["comparisons"]:
        entry["overall_fit"] = r["comparisons"]["overall"]
    json_out.append(entry)

(OUTPUT_DIR / "results.json").write_text(
    json.dumps(json_out, indent=2, ensure_ascii=False), encoding="utf-8"
)
print(f"Saved results.json ({len(json_out)} entries)")

Saved results.json (96 entries)


### Overall statistics

In [14]:
ok = [r for r in all_results if r["status"] == "success"]

total_pop = sum(r["stats"]["total_population"] for r in ok)
total_hh  = sum(r["stats"]["total_households"] for r in ok)

corrs = [r["comparisons"]["overall"]["correlation"]
         for r in ok if r["comparisons"] and "overall" in r["comparisons"]]
mapes = [r["comparisons"]["overall"].get("mape", 0)
         for r in ok if r["comparisons"] and "overall" in r["comparisons"]]

print(f"Successful areas:    {len(ok)}")
print(f"Total population:    {total_pop:,}")
print(f"Total households:    {total_hh:,}")
print(f"Median correlation:  {np.median(corrs):.4f}")
print(f"Mean MAPE:           {np.mean(mapes):.1f}%")
print()
print("Quality grades (by correlation):")
for label, lo, hi in [("A (≥0.99)", 0.99, 2), ("B (0.98–0.99)", 0.98, 0.99),
                       ("C (0.97–0.98)", 0.97, 0.98), ("D/F (<0.97)", 0, 0.97)]:
    n = sum(1 for c in corrs if lo <= c < hi)
    print(f"  {label}: {n} areas ({100*n/len(corrs):.0f}%)")

Successful areas:    95
Total population:    600,541
Total households:    289,504
Median correlation:  0.9961
Mean MAPE:           10.4%

Quality grades (by correlation):
  A (≥0.99): 63 areas (66%)
  B (0.98–0.99): 12 areas (13%)
  C (0.97–0.98): 10 areas (11%)
  D/F (<0.97): 10 areas (11%)
